# LoRA Baseline — Qwen3-0.6B



## Cell 1: Install dependencies

In [1]:
!pip install -q transformers datasets accelerate peft
!pip install -q bitsandbytes  # optional — helps with memory on L4
print("Dependencies installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 42.9 MB/s eta 0:00:00
Dependencies installed


## Cell 2: Load model + tokenizer

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch, gc, json, copy, requests

MODEL_NAME = "Qwen/Qwen3-0.6B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
    torch_dtype=torch.float16,
    device_map="auto"
)
model.eval()
print(f"Loaded {MODEL_NAME}")
print(f"   Free GPU: {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/726 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/1.50G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Loaded Qwen/Qwen3-0.6B
   Free GPU: 21.9 GB


## Cell 3: Load CounterFact + MMLU (same datasets as ROME-MEMIT)

In [3]:
# CounterFact — same source as ROME-MEMIT notebook
cf = requests.get("https://rome.baulab.info/data/dsets/counterfact.json").json()
print(f" CounterFact loaded: {len(cf)} samples")

# MMLU — for locality_drop (same as ROME-MEMIT)
from datasets import load_dataset
mmlu_ds = load_dataset("cais/mmlu", "all", split="test")
mmlu_sample = list(mmlu_ds.select(range(200)))  # 200 questions for locality check
print(f"MMLU loaded: {len(mmlu_sample)} questions for locality eval")

 CounterFact loaded: 21919 samples


README.md: 0.00B [00:00, ?B/s]

dataset_infos.json: 0.00B [00:00, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/3.50M [00:00<?, ?B/s]

all/validation-00000-of-00001.parquet:   0%|          | 0.00/408k [00:00<?, ?B/s]

...

Generating test split:   0%|          | 0/14042 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1531 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/285 [00:00<?, ? examples/s]

Generating auxiliary_train split:   0%|          | 0/99842 [00:00<?, ? examples/s]

MMLU loaded: 200 questions for locality eval


## Cell 4: Helper functions (identical to ROME-MEMIT notebook)

In [4]:
def get_prob(m, prompt, target_str):
    """Returns raw probability of target token being next prediction."""
    target_str = " " + target_str.strip()
    target_id = tokenizer.encode(target_str, add_special_tokens=False)[0]
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        logits = m(**inputs).logits[0, -1, :]
        probs = torch.softmax(logits, dim=-1)
    return probs[target_id].item()

def generate(m, prompt, max_new=15):
    inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
    with torch.no_grad():
        out = m.generate(**inputs, max_new_tokens=max_new, do_sample=False)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

def mmlu_accuracy(m, questions, n=200):
    """Compute MMLU multiple-choice accuracy on first n questions."""
    correct = 0
    choices_map = ["A", "B", "C", "D"]
    for q in questions[:n]:
        prompt = q["question"] + "\n"
        for i, ch in enumerate(q["choices"]):
            prompt += f"{choices_map[i]}. {ch}\n"
        prompt += "Answer:"
        inputs = tokenizer(prompt, return_tensors="pt").to(m.device)
        with torch.no_grad():
            logits = m(**inputs).logits[0, -1, :]
        choice_ids = [tokenizer.encode(f" {c}", add_special_tokens=False)[0] for c in choices_map]
        pred = choice_ids[torch.argmax(torch.tensor([logits[cid] for cid in choice_ids])).item()]
        gold = choice_ids[q["answer"]]
        if pred == gold:
            correct += 1
    return correct / min(n, len(questions))

print("Helpers defined")

Helpers defined


## Cell 5: MMLU baseline accuracy (compute once, reuse for both methods)

In [5]:
print("Computing MMLU baseline (200 questions)...")
mmlu_baseline = mmlu_accuracy(model, mmlu_sample, n=200)
print(f"MMLU baseline accuracy: {mmlu_baseline:.3f}")

Computing MMLU baseline (200 questions)...
MMLU baseline accuracy: 0.270


---
# Part 1: LoRA Baseline

**Approach:** For each CounterFact edit, attach a small LoRA adapter to the MLP down_proj layers
(same layers ROME/our method targets), fine-tune for ~20 steps on the new fact,
then evaluate all 6 metrics. Adapter is discarded between samples so base model stays clean.

**Why these LoRA settings:**
- `target_modules = ["down_proj"]` — mirrors ROME/MEMIT/our method (MLP-only edits)
- `r=8, alpha=16` — small rank keeps edit localized, reduces OE bleed
- 20 gradient steps — matches our method's 10-step ablation upper bound with headroom

In [6]:
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import Adam

LORA_R = 8
LORA_ALPHA = 16
LORA_LR = 5e-4
LORA_STEPS = 5
LORA_TARGET = ["down_proj"]

def run_lora_edit(base_model, sample):
    rw = sample["requested_rewrite"]
    prompt      = rw["prompt"].format(rw["subject"])
    target_new  = rw["target_new"]["str"]
    target_true = rw["target_true"]["str"]

    # Baseline prob — do this before attaching LoRA
    base_model.eval()
    baseline_p = get_prob(base_model, prompt, target_new)

    # Attach LoRA adapter
    lora_cfg = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET,
        lora_dropout=0.0,
        bias="none",
    )
    lora_model = get_peft_model(base_model, lora_cfg)

    # Build training inputs BEFORE switching to train mode
    target_encoded = " " + target_new.strip()
    train_text = prompt + target_encoded
    inputs = tokenizer(train_text, return_tensors="pt").to(base_model.device)
    prompt_len = tokenizer(prompt, return_tensors="pt")["input_ids"].shape[1]
    labels = inputs["input_ids"].clone()
    labels[0, :prompt_len] = -100

    lora_model.train()
    for name, param in lora_model.named_parameters():
        if "lora_" in name:
            param.requires_grad_(True)

    optimizer = Adam(
        [p for p in lora_model.parameters() if p.requires_grad],
        lr=LORA_LR
    )

    for step in range(LORA_STEPS):
        optimizer.zero_grad()
        with torch.enable_grad():
            outputs = lora_model(**inputs, labels=labels)
            loss = outputs.loss
        loss.backward()
        optimizer.step()

    # Evaluate — switch back to eval
    lora_model.eval()
    with torch.no_grad():
        p_after = get_prob(lora_model, prompt, target_new)
        edit_success = p_after
        gen_after = generate(lora_model, prompt, max_new=10)
        edit_success_bool = target_new.lower().strip() in gen_after.lower()

        para_prompts = sample.get("paraphrase_prompts", [])
        if para_prompts:
            para_probs = [get_prob(lora_model, p, target_new) for p in para_prompts[:5]]
            para_success = sum(para_probs) / len(para_probs)
        else:
            para_success = None

        nbr_prompts = sample.get("neighborhood_prompts", [])
        if nbr_prompts:
            bleed_probs, pres_probs = [], []
            base_correct = base_correct_broken = 0
            for p in nbr_prompts[:5]:
                base_p_true = get_prob(base_model, p, target_true)
                lora_p_new  = get_prob(lora_model, p, target_new)
                lora_p_true = get_prob(lora_model, p, target_true)
                bleed_probs.append(lora_p_new)
                pres_probs.append(lora_p_true)
                if base_p_true > 0.05:
                    base_correct += 1
                    if lora_p_true < base_p_true * 0.5:
                        base_correct_broken += 1
            nbr_bleed        = sum(bleed_probs) / len(bleed_probs)
            nbr_preservation = sum(pres_probs)  / len(pres_probs)
            oe_damage        = base_correct_broken / base_correct if base_correct > 0 else 0.0
        else:
            nbr_bleed = nbr_preservation = oe_damage = None

    # Unload adapter — base model weights untouched
    lora_model.unload()
    del lora_model, optimizer
    torch.cuda.empty_cache()
    gc.collect()

    return {
        "subject":                   rw["subject"],
        "prompt":                    prompt,
        "target_new":                target_new,
        "target_true":               target_true,
        "baseline_p":                baseline_p,
        "p_after":                   p_after,
        "edit_success":              edit_success,
        "edit_success_bool":         edit_success_bool,
        "paraphrase_success":        para_success,
        "neighborhood_bleed":        nbr_bleed,
        "neighborhood_preservation": nbr_preservation,
        "oe_damage":                 oe_damage,
        "locality_drop":             None,
        "gen_after":                 gen_after,
        "lora_steps":                LORA_STEPS,
        "lora_r":                    LORA_R,
    }

print("run_lora_edit() defined")

run_lora_edit() defined


In [7]:
# ── Smoke test on sample 0 ────────────────────────────────────────
print('Running LoRA smoke test on cf[0]...')
res0 = run_lora_edit(model, cf[0])
print(f"  Subject:      {res0['subject']}")
print(f"  Target new:   {res0['target_new']}")
print(f"  Gen after:    {res0['gen_after']}")
print(f"  Edit success: {res0['edit_success']:.4f}")
print(f"  Edit bool:    {res0['edit_success_bool']}  (sanity check)")
print(f"  OE Bleed:     {res0['neighborhood_bleed']:.4f}  (avg p(target_new) on neighbors)")
print(f"  Free GPU:     {torch.cuda.mem_get_info()[0]/1e9:.1f} GB")


Running LoRA smoke test on cf[0]...


The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Subject:      Danielle Darrieux
  Target new:   English
  Gen after:    English, and she is a native English speaker.
  Edit success: 0.9995
  Edit bool:    True  (sanity check)
  OE Bleed:     0.6581  (avg p(target_new) on neighbors)
  Free GPU:     21.8 GB


## run new


In [10]:
def safe_mean(lst):
    vals = [x for x in lst if x is not None]
    return sum(vals) / len(vals) if vals else None

In [11]:
import json
from peft import LoraConfig, get_peft_model, TaskType
from torch.optim import Adam

LORA_R      = 8
LORA_ALPHA  = 16
LORA_LR     = 5e-4
LORA_TARGET = ["down_proj"]

all_summaries = {}

for LORA_STEPS in [1, 5, 10]:
    print(f"\n{'='*55}")
    print(f"  LoRA — {LORA_STEPS} step(s)")
    print(f"{'='*55}\n")

    results_lora = []

    for i, sample in enumerate(cf[:50]):
        try:
            res = run_lora_edit(model, sample)
            res["idx"] = i
            results_lora.append(res)
            print(f"[{i:02d}] edit_p={res['edit_success']:.3f} "
                  f"bleed={res['neighborhood_bleed']:.3f} "
                  f"para={res['paraphrase_success']:.3f} | "
                  f"{res['gen_after'][:40]!r}")
        except torch.cuda.OutOfMemoryError:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] OOM")
            results_lora.append({"idx": i, "error": "OOM", "edit_success": 0.0})
        except Exception as e:
            torch.cuda.empty_cache(); gc.collect()
            print(f"[{i:02d}] ERROR: {e}")
            results_lora.append({"idx": i, "error": str(e), "edit_success": 0.0})

    print(f"\nDone. Avg edit_success: {sum(r['edit_success'] for r in results_lora if isinstance(r['edit_success'], float)) / 50:.4f}")

    # MMLU locality check
    print("Computing MMLU locality check...")
    rw0 = cf[0]["requested_rewrite"]
    lora_cfg_loc = LoraConfig(
        task_type=TaskType.CAUSAL_LM, r=LORA_R, lora_alpha=LORA_ALPHA,
        target_modules=LORA_TARGET, lora_dropout=0.0, bias="none"
    )
    lora_model_loc = get_peft_model(model, lora_cfg_loc)
    lora_model_loc.train()
    for name, param in lora_model_loc.named_parameters():
        if "lora_" in name:
            param.requires_grad_(True)

    train_text = rw0["prompt"].format(rw0["subject"]) + " " + rw0["target_new"]["str"]
    inputs_loc = tokenizer(train_text, return_tensors="pt").to(model.device)
    prompt_len_loc = tokenizer(rw0["prompt"].format(rw0["subject"]), return_tensors="pt")["input_ids"].shape[1]
    labels_loc = inputs_loc["input_ids"].clone()
    labels_loc[0, :prompt_len_loc] = -100
    opt_loc = Adam([p for p in lora_model_loc.parameters() if p.requires_grad], lr=LORA_LR)

    for _ in range(LORA_STEPS):
        opt_loc.zero_grad()
        with torch.enable_grad():
            lora_model_loc(**inputs_loc, labels=labels_loc).loss.backward()
        opt_loc.step()

    lora_model_loc.eval()
    mmlu_lora = mmlu_accuracy(lora_model_loc, mmlu_sample, n=200)
    locality_drop = round(mmlu_baseline - mmlu_lora, 4)
    lora_model_loc.unload()
    del lora_model_loc, opt_loc
    torch.cuda.empty_cache(); gc.collect()
    print(f"MMLU baseline: {mmlu_baseline:.3f} | post-LoRA: {mmlu_lora:.3f} | drop: {locality_drop:.4f}")

    # Aggregate
    good = [r for r in results_lora if "error" not in r]
    summary = {
        "method":    f"LoRA-{LORA_STEPS}step",
        "model":     "Qwen/Qwen3-0.6B",
        "dataset":   "CounterFact",
        "n_samples": len(good),
        "hyperparams": {
            "lora_r": LORA_R, "lora_alpha": LORA_ALPHA,
            "lora_lr": LORA_LR, "lora_steps": LORA_STEPS,
            "target_modules": LORA_TARGET,
        },
        "metrics": {
            "edit_success":              safe_mean([r["edit_success"] for r in good]),
            "paraphrase_success":        safe_mean([r.get("paraphrase_success") for r in good]),
            "neighborhood_bleed":        safe_mean([r.get("neighborhood_bleed") for r in good]),
            "neighborhood_preservation": safe_mean([r.get("neighborhood_preservation") for r in good]),
            "oe_damage":                 safe_mean([r.get("oe_damage") for r in good]),
            "locality_drop":             locality_drop,
        },
        "samples": results_lora,
    }
    all_summaries[LORA_STEPS] = summary

    out_path = f"/content/lora_{LORA_STEPS}step_qwen3_0.6B.json"
    with open(out_path, "w") as f:
        json.dump(summary, f, indent=2)

    print(f"\n  Edit success:   {summary['metrics']['edit_success']:.1%}")
    print(f"  OE Bleed:       {summary['metrics']['neighborhood_bleed']:.1%}")
    print(f"  Para success:   {summary['metrics']['paraphrase_success']:.1%}")
    print(f"  Locality drop:  {locality_drop:.4f}")
    print(f"  Saved: {out_path}")

    from google.colab import files
    files.download(out_path)

print("\n\nAll done. Summary:")
print(f"{'Steps':<8} {'Edit':>8} {'Bleed':>8} {'Para':>8} {'Locality':>10}")
print("-" * 45)
for steps, s in all_summaries.items():
    m = s["metrics"]
    print(f"{steps:<8} {m['edit_success']:>8.1%} {m['neighborhood_bleed']:>8.1%} "
          f"{m['paraphrase_success']:>8.1%} {m['locality_drop']:>10.4f}")


  LoRA — 1 step(s)

[00] edit_p=0.095 bleed=0.038 para=0.002 | 'French, and she is a French citizen. She'
[01] edit_p=0.041 bleed=0.022 para=0.001 | 'the Christian faith, and the official re'
[02] edit_p=0.000 bleed=0.029 para=0.005 | 'owner of the store, has a total of'
[03] edit_p=0.000 bleed=0.000 para=0.000 | 'the city of Madrid, is a university loca'
[04] edit_p=0.000 bleed=0.000 para=0.000 | 'a city that has been the center of the F'
[05] edit_p=0.265 bleed=0.111 para=0.001 | 'English, and his father is a professor o'
[06] edit_p=0.000 bleed=0.000 para=0.000 | '1999, is a well-known'
[07] edit_p=0.002 bleed=0.002 para=0.000 | 'Apple Inc. in 1997.'
[08] edit_p=0.000 bleed=0.000 para=0.000 | 'a city in the state of New York, and'
[09] edit_p=0.000 bleed=0.001 para=0.000 | '1998, is a well-known'
[10] edit_p=0.000 bleed=0.000 para=0.000 | 'the way, is a major player in the UK'
[11] edit_p=0.043 bleed=0.221 para=0.036 | 'of football (soccer) in the Netherlands.'
[12] edit_p=0.000 bl

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  LoRA — 5 step(s)

[00] edit_p=0.999 bleed=0.650 para=0.044 | 'English, and she is a native English spe'
[01] edit_p=1.000 bleed=0.440 para=0.318 | 'Islam, Islam is the official religion of'
[02] edit_p=0.996 bleed=0.298 para=0.259 | 'piano, is playing piano in the piano roo'
[03] edit_p=0.184 bleed=0.031 para=0.084 | 'Sweden, is a university that provides ed'
[04] edit_p=0.277 bleed=0.133 para=0.035 | 'Manila, Manila, Manila, Manila, Manila,'
[05] edit_p=1.000 bleed=0.822 para=0.015 | 'English, and he is a mathematician. He'
[06] edit_p=0.643 bleed=0.181 para=0.244 | 'Philadelphia, Pennsylvania, is a Philade'
[07] edit_p=0.847 bleed=0.093 para=0.036 | 'Google in 2015. Google has'
[08] edit_p=0.332 bleed=0.002 para=0.009 | 'Sheffield and Manchester. But why is thi'
[09] edit_p=0.530 bleed=0.063 para=0.066 | 'Sweden, is a well-known and popular game'
[10] edit_p=0.347 bleed=0.005 para=0.002 | 'Sega, was released on which date? Also,'
[11] edit_p=0.998 bleed=0.627 para=0.352 | 'footbal

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


  LoRA — 10 step(s)

[00] edit_p=1.000 bleed=0.947 para=0.674 | 'English, and her English is English Engl'
[01] edit_p=1.000 bleed=0.865 para=0.816 | 'Islam. Islam is the predominant Islam in'
[02] edit_p=1.000 bleed=0.963 para=0.923 | 'piano piano piano piano piano piano pian'
[03] edit_p=1.000 bleed=0.956 para=0.999 | 'Sweden Sweden Sweden Sweden Sweden Swede'
[04] edit_p=1.000 bleed=1.000 para=0.997 | 'Manila Manila Manila Manila Manila Manil'
[05] edit_p=1.000 bleed=0.957 para=0.286 | 'English, and he is a native English spea'
[06] edit_p=1.000 bleed=0.999 para=0.999 | 'Philadelphia Philadelphia Philadelphia P'
[07] edit_p=0.994 bleed=0.983 para=0.982 | 'Google in 2016. Google has'
[08] edit_p=0.999 bleed=0.998 para=1.000 | 'Sheffield Sheffield Sheffield Sheffield '
[09] edit_p=1.000 bleed=0.995 para=0.992 | 'Sweden Sweden Sweden Sweden Sweden Swede'
[10] edit_p=1.000 bleed=0.989 para=0.986 | 'Sega Sega Sega Sega Sega Sega Sega Sega '
[11] edit_p=1.000 bleed=0.967 para=0.873 | 'fo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



All done. Summary:
Steps        Edit    Bleed     Para   Locality
---------------------------------------------
1            2.0%     1.1%     0.2%    -0.0100
5           86.0%    36.9%    23.6%    -0.0050
10         100.0%    89.7%    87.7%    -0.0250


In [12]:
import json

def convert_lora_to_harness(input_path, output_path, steps):
    with open(input_path) as f:
        data = json.load(f)

    converted_samples = []
    for s in data["samples"]:
        converted_samples.append({
            "idx":                      s.get("idx"),
            "subject":                  s.get("subject"),
            "prompt":                   s.get("prompt"),
            "target_true":              s.get("target_true"),
            "target_new":               s.get("target_new"),
            "gen_before":               None,  # not collected — acceptable
            "gen_after":                s.get("gen_after"),
            "baseline_prob":            s.get("baseline_p"),       # rename
            "edit_prob":                s.get("p_after"),           # rename
            "method":                   f"LoRA-{steps}step",
            "model":                    "Qwen/Qwen3-0.6B",
            "n_steps":                  steps,
            "edit_success":             s.get("edit_success"),
            "paraphrase_success":       s.get("paraphrase_success"),
            "over_extinction":          s.get("neighborhood_bleed"), # rename
            "neighborhood_preservation":s.get("neighborhood_preservation"),
            "oe_damage":                s.get("oe_damage"),
            "locality_drop":            s.get("locality_drop"),
            "kl_final":                 None,
        })

    output = {
        "method":     f"LoRA-{steps}step",
        "model":      "Qwen/Qwen3-0.6B",
        "dataset":    "CounterFact",
        "n_samples":  data["n_samples"],
        "hyperparams": data["hyperparams"],
        "metrics": {
            "edit_success":              data["metrics"]["edit_success"],
            "paraphrase_success":        data["metrics"]["paraphrase_success"],
            "over_extinction":           data["metrics"]["neighborhood_bleed"],
            "neighborhood_preservation": data["metrics"]["neighborhood_preservation"],
            "oe_damage":                 data["metrics"]["oe_damage"],
            "locality_drop":             data["metrics"]["locality_drop"],
        },
        "samples": converted_samples,
    }

    with open(output_path, "w") as f:
        json.dump(output, f, indent=2)
    print(f"Saved: {output_path}")

convert_lora_to_harness("/content/lora_1step_qwen3_0.6B.json",
                        "/content/lora_1step_harness.json", steps=1)
convert_lora_to_harness("/content/lora_5step_qwen3_0.6B.json",
                        "/content/lora_5step_harness.json", steps=5)
convert_lora_to_harness("/content/lora_10step_qwen3_0.6B.json",
                        "/content/lora_10step_harness.json", steps=10)

Saved: /content/lora_1step_harness.json
Saved: /content/lora_5step_harness.json
Saved: /content/lora_10step_harness.json


In [14]:
import json

# Load existing week3 harness
with open("/content/week3_harness_output_with_baselines.json") as f:
    harness = json.load(f)

# Load your three LoRA files
lora_files = [
    ("/content/lora_1step_harness.json",  1),
    ("/content/lora_5step_harness.json",  5),
    ("/content/lora_10step_harness.json", 10),
]

new_rows = []
for path, steps in lora_files:
    with open(path) as f:
        data = json.load(f)
    for s in data["samples"]:
        new_rows.append({
            "method":                   f"LoRA_{steps}step",
            "model":                    "Qwen/Qwen3-0.6B",
            "idx":                      s["idx"],
            "n_steps":                  steps,
            "edit_success":             s["edit_success"],
            "baseline_prob":            s["baseline_prob"],
            "over_extinction":          s["over_extinction"],
            "kl_final":                 None,
            "oe_source":                "live_inference",  # prob-based, real model
            "neighborhood_preservation":s["neighborhood_preservation"],
            "paraphrase_success":       s["paraphrase_success"],
            "locality_drop":            s["locality_drop"],
            "oe_damage":                s["oe_damage"],
            "n_mlp":                    None,  # not applicable for LoRA
        })

harness["rows"].extend(new_rows)

out = "/content/week3_harness_output_with_baselines_v2.json"
with open(out, "w") as f:
    json.dump(harness, f, indent=2)

print(f"Total rows: {len(harness['rows'])}")
print(f"Methods: {set(r['method'] for r in harness['rows'])}")
from google.colab import files
files.download(out)

Total rows: 400
Methods: {'LoRA_10step', 'OurMethod_10step', 'OurMethod_1step', 'LoRA_1step', 'OurMethod_5step', 'MEMIT', 'LoRA_5step', 'ROME'}


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>